In [ ]:
# ============================================
# CELL 1: IMPORT LIBRARIES
# ============================================

import tensorflow as tf
import numpy as np
import re
import time

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [ ]:
# ============================================
# CELL 2: TRAINING DATA
# ============================================

# English sentences
english_sentences = [
    "hello",
    "how are you",
    "i am fine",
    "good morning",
    "good night",
    "thank you",
    "what is your name",
    "my name is john",
    "i love you",
    "where are you",
    "i am happy",
    "i am sad",
    "see you",
    "goodbye",
    "how old are you"
]

# Corresponding French translations
french_sentences = [
    "bonjour",
    "comment allez vous",
    "je vais bien",
    "bonjour",
    "bonne nuit",
    "merci",
    "quel est votre nom",
    "mon nom est john",
    "je vous aime",
    "ou etes vous",
    "je suis heureux",
    "je suis triste",
    "a bientot",
    "au revoir",
    "quel age avez vous"
]

print("Number of training examples:", len(english_sentences))

Number of training examples: 15


In [ ]:
# ============================================
# CELL 3: ADD SPECIAL TOKENS
# ============================================

french_sentences_with_tokens = [
    "<start> " + sentence + " <end>"
    for sentence in french_sentences
]

for i in range(3):
    print(english_sentences[i], " --> ", french_sentences_with_tokens[i])

hello  -->  <start> bonjour <end>
how are you  -->  <start> comment allez vous <end>
i am fine  -->  <start> je vais bien <end>


In [ ]:
# ============================================
# CELL 4: TOKENIZATION
# ============================================

def tokenize_sentences(sentences):
    tokenizer = tf.keras.preprocessing.text.Tokenizer(
        filters='',
        lower=True,
        oov_token="<unk>"
    )

    tokenizer.fit_on_texts(sentences)

    sequences = tokenizer.texts_to_sequences(sentences)

    return tokenizer, sequences


# Create English tokenizer
english_tokenizer, english_sequences = tokenize_sentences(
    english_sentences
)

# Create French tokenizer
french_tokenizer, french_sequences = tokenize_sentences(
    french_sentences_with_tokens
)


english_vocab_size = len(english_tokenizer.word_index) + 1
french_vocab_size = len(french_tokenizer.word_index) + 1

print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

English vocabulary size: 26
French vocabulary size: 32


In [ ]:
# ============================================
# CELL 5: PADDING
# ============================================

max_encoder_length = max(
    len(sequence)
    for sequence in english_sequences
)

max_decoder_length = max(
    len(sequence)
    for sequence in french_sequences
)

english_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    english_sequences,
    maxlen=max_encoder_length,
    padding="post"
)

french_sequences = tf.keras.preprocessing.sequence.pad_sequences(
    french_sequences,
    maxlen=max_decoder_length,
    padding="post"
)

print("English sequences:")
print(english_sequences)

print("\nFrench sequences:")
print(french_sequences)

English sequences:
[[10  0  0  0]
 [ 6  4  2  0]
 [ 3  5 11  0]
 [ 7 12  0  0]
 [ 7 13  0  0]
 [14  2  0  0]
 [15  8 16  9]
 [17  9  8 18]
 [ 3 19  2  0]
 [20  4  2  0]
 [ 3  5 21  0]
 [ 3  5 22  0]
 [23  2  0  0]
 [24  0  0  0]
 [ 6 25  4  2]]

French sequences:
[[ 2  6  3  0  0  0]
 [ 2 11 12  4  3  0]
 [ 2  5 13 14  3  0]
 [ 2  6  3  0  0  0]
 [ 2 15 16  3  0  0]
 [ 2 17  3  0  0  0]
 [ 2  7  8 18  9  3]
 [ 2 19  9  8 20  3]
 [ 2  5  4 21  3  0]
 [ 2 22 23  4  3  0]
 [ 2  5 10 24  3  0]
 [ 2  5 10 25  3  0]
 [ 2 26 27  3  0  0]
 [ 2 28 29  3  0  0]
 [ 2  7 30 31  4  3]]


In [ ]:
# ============================================
# CELL 6: DECODER INPUT AND TARGET
# ============================================

decoder_inputs = french_sequences[:, :-1]

decoder_targets = french_sequences[:, 1:]

print("Decoder input:")
print(decoder_inputs)

print("\nDecoder target:")
print(decoder_targets)

Decoder input:
[[ 2  6  3  0  0]
 [ 2 11 12  4  3]
 [ 2  5 13 14  3]
 [ 2  6  3  0  0]
 [ 2 15 16  3  0]
 [ 2 17  3  0  0]
 [ 2  7  8 18  9]
 [ 2 19  9  8 20]
 [ 2  5  4 21  3]
 [ 2 22 23  4  3]
 [ 2  5 10 24  3]
 [ 2  5 10 25  3]
 [ 2 26 27  3  0]
 [ 2 28 29  3  0]
 [ 2  7 30 31  4]]

Decoder target:
[[ 6  3  0  0  0]
 [11 12  4  3  0]
 [ 5 13 14  3  0]
 [ 6  3  0  0  0]
 [15 16  3  0  0]
 [17  3  0  0  0]
 [ 7  8 18  9  3]
 [19  9  8 20  3]
 [ 5  4 21  3  0]
 [22 23  4  3  0]
 [ 5 10 24  3  0]
 [ 5 10 25  3  0]
 [26 27  3  0  0]
 [28 29  3  0  0]
 [ 7 30 31  4  3]]


In [ ]:
# ============================================
# CELL 7: POSITIONAL ENCODING
# ============================================

# ============================================
# FIXED POSITIONAL ENCODING
# ============================================

def positional_encoding(length, depth):

    # Half of the dimensions use sine
    # and the other half use cosine

    depth = depth // 2

    # Create positions:
    #
    # [0]
    # [1]
    # [2]
    # ...
    #
    # Shape:
    # (length, 1)

    positions = tf.range(
        length,
        dtype=tf.float32
    )[:, tf.newaxis]

    # Create depth indices
    #
    # [0, 1, 2, 3, ...]

    depths = tf.range(
        depth,
        dtype=tf.float32
    )[tf.newaxis, :]

    # Scale depth values

    depths = depths / tf.cast(
        depth,
        tf.float32
    )

    # Calculate angle rates

    angle_rates = 1 / (
        10000 ** depths
    )

    # Position × angle rate

    angle_rads = positions * angle_rates

    # Sine for first half
    # Cosine for second half

    pos_encoding = tf.concat(
        [
            tf.sin(angle_rads),
            tf.cos(angle_rads)
        ],
        axis=-1
    )

    return pos_encoding

In [ ]:
# ============================================
# CELL 8: EMBEDDING + POSITION
# ============================================

class PositionalEmbedding(tf.keras.layers.Layer):

    def __init__(self, vocab_size, d_model):

        super().__init__()

        self.d_model = d_model

        # Converts token IDs into vectors
        self.embedding = tf.keras.layers.Embedding(
            vocab_size,
            d_model
        )

    def call(self, x):

        length = tf.shape(x)[1]

        # Word/token embedding
        x = self.embedding(x)

        # Scale embeddings
        x *= tf.math.sqrt(
            tf.cast(self.d_model, tf.float32)
        )

        # Add positional information
        x += positional_encoding(
            length,
            self.d_model
        )

        return x

In [ ]:
# ============================================
# CELL 9: PADDING MASK
# ============================================

def create_padding_mask(sequence):

    # Padding token has ID 0
    mask = tf.cast(
        tf.equal(sequence, 0),
        tf.float32
    )

    # Shape:
    # (batch, 1, 1, sequence_length)

    return mask[:, tf.newaxis, tf.newaxis, :]

In [ ]:
# ============================================
# CELL 10: LOOK-AHEAD MASK
# ============================================

def create_look_ahead_mask(size):

    # Upper triangular matrix
    # prevents looking at future tokens

    mask = 1 - tf.linalg.band_part(
        tf.ones((size, size)),
        -1,
        0
    )

    return mask

In [ ]:
# ============================================
# CELL 11: MULTI-HEAD ATTENTION
# ============================================

class MultiHeadAttention(tf.keras.layers.Layer):

    def __init__(self, d_model, num_heads):

        super().__init__()

        self.num_heads = num_heads

        self.d_model = d_model

        assert d_model % num_heads == 0

        self.depth = d_model // num_heads

        # Linear transformations
        # used to create Q, K and V

        self.wq = tf.keras.layers.Dense(d_model)

        self.wk = tf.keras.layers.Dense(d_model)

        self.wv = tf.keras.layers.Dense(d_model)

        # Final transformation
        self.dense = tf.keras.layers.Dense(d_model)


    def split_heads(self, x, batch_size):

        # x shape:
        # (batch, sequence_length, d_model)

        x = tf.reshape(
            x,
            (
                batch_size,
                -1,
                self.num_heads,
                self.depth
            )
        )

        # Change dimensions:
        #
        # (batch, heads, sequence_length, depth)

        return tf.transpose(x, [0, 2, 1, 3])


    def call(self, v, k, q, mask=None):

        batch_size = tf.shape(q)[0]

        # Create Q, K and V

        q = self.wq(q)

        k = self.wk(k)

        v = self.wv(v)

        # Split into multiple attention heads

        q = self.split_heads(q, batch_size)

        k = self.split_heads(k, batch_size)

        v = self.split_heads(v, batch_size)

        # ---------------------------------------
        # Scaled dot-product attention
        # ---------------------------------------

        attention_scores = tf.matmul(
            q,
            k,
            transpose_b=True
        )

        # Scale by sqrt(depth)

        scale = tf.math.sqrt(
            tf.cast(self.depth, tf.float32)
        )

        attention_scores /= scale

        # Apply mask if provided

        if mask is not None:

            attention_scores += (
                mask * -1e9
            )

        # Softmax converts scores to probabilities

        attention_weights = tf.nn.softmax(
            attention_scores,
            axis=-1
        )

        # Weighted sum of V

        output = tf.matmul(
            attention_weights,
            v
        )

        # ---------------------------------------
        # Combine attention heads
        # ---------------------------------------

        output = tf.transpose(
            output,
            [0, 2, 1, 3]
        )

        output = tf.reshape(
            output,
            (
                batch_size,
                -1,
                self.d_model
            )
        )

        # Final linear layer

        output = self.dense(output)

        return output

In [ ]:
# ============================================
# CELL 12: FEED FORWARD NETWORK
# ============================================

class FeedForward(tf.keras.layers.Layer):

    def __init__(self, d_model, dff):

        super().__init__()

        self.network = tf.keras.Sequential([

            tf.keras.layers.Dense(
                dff,
                activation="relu"
            ),

            tf.keras.layers.Dense(
                d_model
            )
        ])


    def call(self, x):

        return self.network(x)

In [ ]:
# ============================================
# CELL 13: ENCODER LAYER
# ============================================

class EncoderLayer(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        dff,
        dropout_rate=0.1
    ):

        super().__init__()

        self.attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = FeedForward(
            d_model,
            dff
        )

        self.norm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.norm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.dropout1 = tf.keras.layers.Dropout(
            dropout_rate
        )

        self.dropout2 = tf.keras.layers.Dropout(
            dropout_rate
        )


    def call(self, x, training=False, mask=None):

        # Self attention
        attention_output = self.attention(
            x,
            x,
            x,
            mask
        )

        attention_output = self.dropout1(
            attention_output,
            training=training
        )

        # Residual connection + normalization

        x = self.norm1(
            x + attention_output
        )

        # Feed-forward network

        ffn_output = self.ffn(x)

        ffn_output = self.dropout2(
            ffn_output,
            training=training
        )

        # Residual connection + normalization

        x = self.norm2(
            x + ffn_output
        )

        return x

In [ ]:
# ============================================
# CELL 14: ENCODER
# ============================================

class Encoder(tf.keras.layers.Layer):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        dff,
        dropout_rate=0.1
    ):

        super().__init__()

        self.d_model = d_model

        self.embedding = PositionalEmbedding(
            vocab_size,
            d_model
        )

        self.dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

        self.layers_list = [

            EncoderLayer(
                d_model,
                num_heads,
                dff,
                dropout_rate
            )

            for _ in range(num_layers)
        ]


    def call(
        self,
        x,
        training=False,
        mask=None
    ):

        # Token embedding + positional encoding

        x = self.embedding(x)

        x = self.dropout(
            x,
            training=training
        )

        # Pass through every encoder layer

        for layer in self.layers_list:

            x = layer(
                x,
                training=training,
                mask=mask
            )

        return x

In [ ]:
# ============================================
# CELL 15: DECODER LAYER
# ============================================

class DecoderLayer(tf.keras.layers.Layer):

    def __init__(
        self,
        d_model,
        num_heads,
        dff,
        dropout_rate=0.1
    ):

        super().__init__()

        # Decoder self-attention

        self.self_attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        # Encoder-decoder attention

        self.cross_attention = MultiHeadAttention(
            d_model,
            num_heads
        )

        self.ffn = FeedForward(
            d_model,
            dff
        )

        self.norm1 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.norm2 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.norm3 = tf.keras.layers.LayerNormalization(
            epsilon=1e-6
        )

        self.dropout1 = tf.keras.layers.Dropout(
            dropout_rate
        )

        self.dropout2 = tf.keras.layers.Dropout(
            dropout_rate
        )

        self.dropout3 = tf.keras.layers.Dropout(
            dropout_rate
        )


    def call(
        self,
        x,
        encoder_output,
        training=False,
        look_ahead_mask=None,
        padding_mask=None
    ):

        # =====================================
        # 1. MASKED SELF ATTENTION
        # =====================================

        attention1 = self.self_attention(
            x,
            x,
            x,
            look_ahead_mask
        )

        attention1 = self.dropout1(
            attention1,
            training=training
        )

        x = self.norm1(
            x + attention1
        )


        # =====================================
        # 2. CROSS ATTENTION
        # =====================================

        # Query comes from decoder
        #
        # Key and Value come from encoder

        attention2 = self.cross_attention(
            encoder_output,
            encoder_output,
            x,
            padding_mask
        )

        attention2 = self.dropout2(
            attention2,
            training=training
        )

        x = self.norm2(
            x + attention2
        )


        # =====================================
        # 3. FEED FORWARD
        # =====================================

        ffn_output = self.ffn(x)

        ffn_output = self.dropout3(
            ffn_output,
            training=training
        )

        x = self.norm3(
            x + ffn_output
        )

        return x

In [ ]:
# ============================================
# CELL 16: DECODER
# ============================================

class Decoder(tf.keras.layers.Layer):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        num_heads,
        dff,
        dropout_rate=0.1
    ):

        super().__init__()

        self.embedding = PositionalEmbedding(
            vocab_size,
            d_model
        )

        self.dropout = tf.keras.layers.Dropout(
            dropout_rate
        )

        self.layers_list = [

            DecoderLayer(
                d_model,
                num_heads,
                dff,
                dropout_rate
            )

            for _ in range(num_layers)
        ]


    def call(
        self,
        x,
        encoder_output,
        training=False,
        look_ahead_mask=None,
        padding_mask=None
    ):

        x = self.embedding(x)

        x = self.dropout(
            x,
            training=training
        )

        for layer in self.layers_list:

            x = layer(
                x,
                encoder_output,
                training=training,
                look_ahead_mask=look_ahead_mask,
                padding_mask=padding_mask
            )

        return x

In [ ]:
# ============================================
# CELL 17: COMPLETE TRANSFORMER
# ============================================

class Transformer(tf.keras.Model):

    def __init__(
        self,
        encoder_vocab_size,
        decoder_vocab_size,
        d_model,
        num_layers,
        num_heads,
        dff,
        dropout_rate=0.1
    ):

        super().__init__()

        self.encoder = Encoder(
            encoder_vocab_size,
            d_model,
            num_layers,
            num_heads,
            dff,
            dropout_rate
        )

        self.decoder = Decoder(
            decoder_vocab_size,
            d_model,
            num_layers,
            num_heads,
            dff,
            dropout_rate
        )

        # Final layer converts decoder vectors
        # into vocabulary probabilities

        self.final_layer = tf.keras.layers.Dense(
            decoder_vocab_size
        )


    def call(
        self,
        inputs,
        training=False
    ):

        encoder_input, decoder_input = inputs

        # =====================================
        # MASKS
        # =====================================

        encoder_padding_mask = create_padding_mask(
            encoder_input
        )

        decoder_padding_mask = create_padding_mask(
            encoder_input
        )

        decoder_target_padding_mask = create_padding_mask(
            decoder_input
        )

        sequence_length = tf.shape(
            decoder_input
        )[1]

        look_ahead_mask = create_look_ahead_mask(
            sequence_length
        )

        # Combine:
        #
        # future-token mask
        # +
        # padding mask

        look_ahead_mask = tf.maximum(
            look_ahead_mask,
            decoder_target_padding_mask[:, 0, 0, :][:, tf.newaxis, tf.newaxis, :]
        )

        # =====================================
        # ENCODER
        # =====================================

        encoder_output = self.encoder(
            encoder_input,
            training=training,
            mask=encoder_padding_mask
        )

        # =====================================
        # DECODER
        # =====================================

        decoder_output = self.decoder(
            decoder_input,
            encoder_output,
            training=training,
            look_ahead_mask=look_ahead_mask,
            padding_mask=decoder_padding_mask
        )

        # =====================================
        # FINAL VOCABULARY PREDICTION
        # =====================================

        output = self.final_layer(
            decoder_output
        )

        return output

In [ ]:
# ============================================
# CELL 18: MODEL CONFIGURATION
# ============================================

d_model = 128

num_layers = 2

num_heads = 4

dff = 512

dropout_rate = 0.1


transformer = Transformer(
    encoder_vocab_size=english_vocab_size,
    decoder_vocab_size=french_vocab_size,
    d_model=d_model,
    num_layers=num_layers,
    num_heads=num_heads,
    dff=dff,
    dropout_rate=dropout_rate
)

print("Transformer created!")

Transformer created!


In [ ]:
# ============================================
# CELL 19: LOSS FUNCTION
# ============================================

loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)


def loss_function(real, predicted):

    # Calculate loss for every token

    loss = loss_object(
        real,
        predicted
    )

    # Ignore padding tokens

    mask = tf.cast(
        tf.not_equal(real, 0),
        tf.float32
    )

    loss *= mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [ ]:
# ============================================
# CELL 20: OPTIMIZER
# ============================================

optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    beta_1=0.9,
    beta_2=0.98,
    epsilon=1e-9
)

In [ ]:
# ============================================
# CELL 21: TRAINING STEP
# ============================================

@tf.function
def train_step(
    encoder_input,
    decoder_input,
    decoder_target
):

    with tf.GradientTape() as tape:

        # Forward pass

        predictions = transformer(
            (
                encoder_input,
                decoder_input
            ),
            training=True
        )

        # Calculate prediction error

        loss = loss_function(
            decoder_target,
            predictions
        )

    # Calculate gradients

    gradients = tape.gradient(
        loss,
        transformer.trainable_variables
    )

    # Update weights

    optimizer.apply_gradients(
        zip(
            gradients,
            transformer.trainable_variables
        )
    )

    return loss

In [ ]:
# ============================================
# CELL 22: DATASET
# ============================================

BATCH_SIZE = 4

dataset = tf.data.Dataset.from_tensor_slices(
    (
        english_sequences,
        decoder_inputs,
        decoder_targets
    )
)

dataset = dataset.shuffle(
    len(english_sequences)
)

dataset = dataset.batch(
    BATCH_SIZE
)

print("Dataset ready!")

Dataset ready!


In [ ]:
# ============================================
# CELL 23: TRAINING
# ============================================

EPOCHS = 300

for epoch in range(EPOCHS):

    start_time = time.time()

    total_loss = 0

    batches = 0

    for encoder_input, decoder_input, decoder_target in dataset:

        loss = train_step(
            encoder_input,
            decoder_input,
            decoder_target
        )

        total_loss += loss
        batches += 1

    print(
        f"Epoch {epoch + 1}/{EPOCHS} "
        f"Loss: {total_loss / batches:.4f} "
        f"Time: {time.time() - start_time:.2f}s"
    )

/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_4' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_layer_5' (of type EncoderLayer) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/keras/src/layers/layer.py:982: UserWarning: Layer 'encoder_2' (of type Encoder) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Epoch 1/300 Loss: 4.1833 Time: 24.01s
Epoch 2/300 Loss: 3.2571 Time: 0.12s
Epoch 3/300 Loss: 2.9112 Time: 0.12s
Epoch 4/300 Loss: 2.7987 Time: 0.12s
Epoch 5/300 Loss: 2.7145 Time: 0.13s
Epoch 6/300 Loss: 2.4532 Time: 0.11s
Epoch 7/300 Loss: 1.7915 Time: 0.12s
Epoch 8/300 Loss: 1.5469 Time: 0.14s
Epoch 9/300 Loss: 1.0798 Time: 0.11s
Epoch 10/300 Loss: 0.8865 Time: 0.11s
Epoch 11/300 Loss: 0.8037 Time: 0.12s
Epoch 12/300 Loss: 0.6490 Time: 0.11s
Epoch 13/300 Loss: 0.5101 Time: 0.12s
Epoch 14/300 Loss: 0.4187 Time: 0.13s
Epoch 15/300 Loss: 0.2459 Time: 0.11s
Epoch 16/300 Loss: 0.3706 Time: 0.12s
Epoch 17/300 Loss: 0.5371 Time: 0.11s
Epoch 18/300 Loss: 0.2392 Time: 0.11s
Epoch 19/300 Loss: 0.2026 Time: 0.12s
Epoch 20/300 Loss: 0.1592 Time: 0.11s
Epoch 21/300 Loss: 0.1333 Time: 0.12s
Epoch 22/300 Loss: 0.0981 Time: 0.11s
Epoch 23/300 Loss: 0.0785 Time: 0.13s
Epoch 24/300 Loss: 0.0847 Time: 0.11s
Epoch 25/300 Loss: 0.0379 Time: 0.11s
Epoch 26/300 Loss: 0.0625 Time: 0.11s
Epoch 27/300 Loss: 0

In [ ]:
# ============================================
# CELL 24: INFERENCE FUNCTION
# ============================================

def translate(sentence):

    # -----------------------------------------
    # 1. Convert input sentence to token IDs
    # -----------------------------------------

    input_sequence = english_tokenizer.texts_to_sequences(
        [sentence]
    )

    input_sequence = tf.keras.preprocessing.sequence.pad_sequences(
        input_sequence,
        maxlen=max_encoder_length,
        padding="post"
    )

    encoder_input = tf.convert_to_tensor(
        input_sequence,
        dtype=tf.int32
    )


    # -----------------------------------------
    # 2. Get <start> token
    # -----------------------------------------

    start_token = french_tokenizer.word_index[
        "<start>"
    ]

    end_token = french_tokenizer.word_index[
        "<end>"
    ]


    # Start decoder with <start>

    output = [
        start_token
    ]


    # -----------------------------------------
    # 3. Generate one token at a time
    # -----------------------------------------

    for _ in range(max_decoder_length):

        decoder_input = tf.convert_to_tensor(
            [output],
            dtype=tf.int32
        )

        predictions = transformer(
            (
                encoder_input,
                decoder_input
            ),
            training=False
        )

        # Get prediction for LAST token

        next_token_logits = predictions[
            0,
            -1,
            :
        ]

        # Pick token with highest probability

        next_token = tf.argmax(
            next_token_logits
        ).numpy()

        # Add generated token

        output.append(
            int(next_token)
        )

        # Stop when <end> is generated

        if next_token == end_token:

            break


    # -----------------------------------------
    # 4. Convert token IDs back to words
    # -----------------------------------------

    words = []

    for token_id in output:

        word = french_tokenizer.index_word.get(
            token_id,
            ""
        )

        if word not in [
            "<start>",
            "<end>"
        ]:

            words.append(word)


    return " ".join(words)

In [ ]:
# ============================================
# CELL 25: TEST
# ============================================

test_sentences = [
    "hello",
    "how are you",
    "good night",
    "thank you",
    "where are you",
    "i am happy",
    "hello night"
]

for sentence in test_sentences:

    result = translate(sentence)

    print(
        f"English: {sentence}"
    )

    print(
        f"French : {result}"
    )

    print("-" * 40)

English: hello
French : bonjour
----------------------------------------
English: how are you
French : comment allez vous
----------------------------------------
English: good night
French : bonne nuit
----------------------------------------
English: thank you
French : merci
----------------------------------------
English: where are you
French : ou etes vous
----------------------------------------
English: i am happy
French : je suis heureux
----------------------------------------
English: hello night
French : bonne nuit
----------------------------------------
